# Pipeline 2 — IMDb Sentiment Classification

**Owner:** Eman  
**Branch:** `develop`

## What this notebook does
Fine-tunes both `bert-base-uncased` and `distilbert-base-uncased` on the **IMDb** dataset —
a binary sentiment classification task (positive / negative movie reviews).

This corresponds to the IMDb result reported in the DistilBERT paper (Table 2).

Results are saved to `results/` for the final comparison notebook.

## Expected results (from paper Table 2)
| Model | IMDb |
|-------|------|
| BERT-base | 95.63% |
| DistilBERT | 95.79% |

> Note: DistilBERT slightly outperforms BERT on IMDb in the paper — interesting result worth discussing.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import sys
sys.path.append('../src')

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

from train import train_model
from evaluate import evaluate_model
from utils import print_model_info, measure_inference_speed, save_results, count_parameters, model_size_mb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
# IMDb reviews are longer than SST-2 sentences — use 512 tokens
MAX_LENGTH = 512
BATCH_SIZE = 16    # smaller batch size because sequences are longer
EPOCHS     = 3
LR         = 2e-5
SEED       = 42

torch.manual_seed(SEED)

In [ ]:
# ── Load IMDb ──────────────────────────────────────────────────────────────
# 25,000 train / 25,000 test movie reviews
# Labels: 0 = negative, 1 = positive
raw = load_dataset('imdb')
print(f"Train: {len(raw['train'])} | Test: {len(raw['test'])}")
print(f"Sample: {raw['train'][0]['text'][:200]}...")

In [ ]:
# ── Helper: tokenize + build dataloaders ──────────────────────────────────
def get_dataloaders(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True,
                         max_length=MAX_LENGTH, padding='max_length')

    tokenized = raw.map(tokenize, batched=True)

    # DistilBERT has no token_type_ids
    cols = ['input_ids', 'attention_mask', 'label']
    if 'distilbert' not in model_name:
        cols.append('token_type_ids')

    tokenized.set_format('torch', columns=cols)
    tokenized = tokenized.rename_column('label', 'labels')

    train_loader = DataLoader(tokenized['train'], batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(tokenized['test'],  batch_size=BATCH_SIZE)
    return train_loader, test_loader


def run_pipeline(model_name):
    """Fine-tune a model on IMDb, evaluate it, and save results."""
    print(f'\n{"="*50}')
    print(f'Model: {model_name} | Task: IMDb')
    print(f'{"="*50}')

    train_loader, test_loader = get_dataloaders(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    print_model_info(model, model_name)

    history = train_model(model, train_loader, test_loader, DEVICE, epochs=EPOCHS, lr=LR)
    eval_results = evaluate_model(model, test_loader, DEVICE)
    speed = measure_inference_speed(model, test_loader, DEVICE)

    total_params, _ = count_parameters(model)
    results = {
        'model':            model_name,
        'task':             'imdb',
        'val_accuracy':     eval_results['accuracy'],
        'total_parameters': total_params,
        'model_size_mb':    model_size_mb(model),
        'inference_speed':  speed,
        'training_history': history,
    }

    filename = f"imdb_{model_name.replace('/', '_').replace('-', '_')}.json"
    save_results(results, filename)
    print(f"Accuracy: {eval_results['accuracy']}%")
    return results

In [ ]:
# ── Run IMDb: BERT-base ────────────────────────────────────────────────────
bert_imdb = run_pipeline('bert-base-uncased')

In [ ]:
# ── Run IMDb: DistilBERT ───────────────────────────────────────────────────
distilbert_imdb = run_pipeline('distilbert-base-uncased')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
import pandas as pd

rows = [
    {'Model': 'BERT-base (paper)',    'IMDb Accuracy': 95.63},
    {'Model': 'DistilBERT (paper)',   'IMDb Accuracy': 95.79},
    {'Model': 'BERT-base (ours)',     'IMDb Accuracy': bert_imdb['val_accuracy']},
    {'Model': 'DistilBERT (ours)',    'IMDb Accuracy': distilbert_imdb['val_accuracy']},
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))

retention = round(distilbert_imdb['val_accuracy'] / bert_imdb['val_accuracy'] * 100, 1)
print(f'\nDistilBERT retains {retention}% of BERT accuracy on IMDb')